In [1]:
import pyspark, os, sys, json, pandas as pd
from pyspark.sql import SparkSession
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-rdd-drill").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 14:15:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Drill

The file `ol_sample.txt` contains a 0.000001 sample from Open Library all types dump https://openlibrary.org/developers/dumps

- select lines starting with `/type/work`
- extract the JSON payload (last part of the string)
- convert the payload to a python object using the json library
- extract the titles containing the word War
- save these titles to a text file war_titles.txt


## Data Description

The file `data/ol_sample.txt` is a tiny sample (0.000001%) from the **Open Library** all-types dump:  
https://openlibrary.org/developers/dumps

Each line has 5 tab-separated fields:
```
/type/work    /works/OL123W    1    2021-01-01T00:00:00    {"title": "...", ...}
```

- Field 0: `/type/work`, `/type/author`, `/type/edition`, …
- Field 4: JSON payload with metadata (title, subjects, description, …)

**Your task**: use the RDD API to find all work titles containing the word "War" and save them to a text file.

Hint: the `json` library's `json.loads(s)` converts a JSON string to a Python dict.

In [2]:
data = sc.textFile(os.path.join(os.getcwd(), "data/ol_sample.txt"))
data

/app/work/data-engineering-drills/data/ol_sample.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0

In [3]:
# Example using the json library
s = "{\"name\": \"Katy Jakeway\", \"created\": {\"type\": \"/type/datetime\", \"value\": \"2020-09-01T02:40:58.292038\"}, \"last_modified\": {\"type\": \"/type/datetime\", \"value\": \"2020-09-01T02:40:58.292038\"}, \"latest_revision\": 1, \"key\": \"/authors/OL8366070A\", \"type\": {\"key\": \"/type/author\"}, \"revision\": 1}";
d = json.loads(s)
d

{'name': 'Katy Jakeway',
 'created': {'type': '/type/datetime', 'value': '2020-09-01T02:40:58.292038'},
 'last_modified': {'type': '/type/datetime',
  'value': '2020-09-01T02:40:58.292038'},
 'latest_revision': 1,
 'key': '/authors/OL8366070A',
 'type': {'key': '/type/author'},
 'revision': 1}